In [ ]:
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [ ]:
num_classes=10
num_channels=1
img_size=28
patch_size=7
num_patches=(img_size//patch_size)**2
emb_dim=64
attention_heads=2
transformer_blocks=4
dropout_rate=0.1
learning_rate=1e-5

In [ ]:
class GeLU(nn.Module):

    def __init__(self):
        super().__init__()
    def forward(self,x):

        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+(0.044715*(x**3)))))

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(emb_dim,emb_dim*4),
            nn.GELU(),
            nn.Linear(emb_dim*4,emb_dim)
        )
    def forward(self,X):
        return self.network(X)


In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self,emb_dim,num_heads,dropout):
        super().__init__()
        self.query_w=nn.Linear(emb_dim,emb_dim,bias=False)
        self.key_w=nn.Linear(emb_dim,emb_dim,bias=False)
        self.value_w=nn.Linear(emb_dim,emb_dim,bias=False)
        self.head_dim=emb_dim//num_heads
        self.dropout=nn.Dropout(dropout)
        self.num_heads=num_heads
        self.emb_dim=emb_dim
    
    def forward(self,X):
        batch,num_tokens,d_in=X.shape

        query=self.query_w(X)
        keys=self.key_w(X)
        value=self.value_w(X)


        keys=keys.view(batch,num_tokens,self.num_heads,self.head_dim)
        queries=query.view(batch,num_tokens,self.num_heads,self.head_dim)
        values=value.view(batch,num_tokens,self.num_heads,self.head_dim)
        
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores=torch.softmax(attn_scores/(keys.shape[-1]**0.5),dim=-1)
        context_vec = (attn_scores @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(batch, num_tokens, self.emb_dim)
        # optional projection

        return context_vec


In [ ]:
#patch_embedding
class PatchEmbedding(nn.Module):
    def __init__(self,num_channels,emb_dim,patch_size):
        super().__init__()
        self.patch_embed=nn.Conv2d(num_channels,emb_dim,kernel_size=patch_size,stride=patch_size) 

    def forward(self,X):
        
        X=self.patch_embed(X)
        
        X=X.flatten(2)

        X=X.transpose(1,2)
        return X

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.epies=1e-5
        self.shift=nn.Parameter(torch.zeros(emb_dim))
        self.scale=nn.Parameter(torch.ones(emb_dim))
    
    def forward(self,X):

        self.mean=X.mean(dim=-1,keepdim=True)
        self.variance=X.var(-1,keepdim=True)
        norm_x=(X-self.mean)/((self.variance+self.epies)**0.5)

        return norm_x*self.scale+self.shift

In [ ]:
class EncoderTransformerBlock(nn.Module):
    def __init__(self,emb_dim,num_heads,dropout_rate):
        super().__init__()
        self.multiheadattention=MultiHeadAttention(emb_dim,num_heads,dropout_rate)
        self.feedforwardnetwork=FeedForwardNetwork(emb_dim)
        self.layerNorm1=LayerNorm(emb_dim)
        self.layerNorm2=LayerNorm(emb_dim)
        self.dropoutlayer=nn.Dropout(dropout_rate)

    def forward(self,X):
        
        shortcut=X
        X=self.layerNorm1(X)
        X=self.multiheadattention(X)
        X=self.dropoutlayer(X)
        X=X+shortcut
        shortcut=X
        X=self.layerNorm2(X)
        X=self.feedforwardnetwork(X)
        X=self.dropoutlayer(X)
        X=X+shortcut
        return X
        

In [ ]:
class MLP_Head(nn.Module):

    def __init__(self):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(emb_dim,num_classes)
        )
    
    def forward(self,X):
        return self.network(X)

In [ ]:
class VisionTransformer(nn.Module):

    def __init__(self,emb_dim,attention_heads,dropout_rate):
        super().__init__()
        self.patch_embedding=PatchEmbedding(num_channels=num_channels,emb_dim=emb_dim,patch_size=patch_size)
        self.cls_token=nn.Parameter(torch.randn(1,1,emb_dim))
        self.pos_emb=nn.Parameter(torch.randn(1,num_patches+1,emb_dim))
        self.transformer_blocks=nn.Sequential(*[EncoderTransformerBlock(emb_dim,attention_heads,dropout_rate) for _ in range(transformer_blocks)])
        self.mlp_head=MLP_Head()
    def forward(self,X):
        X=self.patch_embedding(X)
        
        class_tokens=self.cls_token.expand(X.size(0),-1,-1)
 
        X=torch.cat((class_tokens,X),dim=1)
    
        X=X+self.pos_emb
        X=self.transformer_blocks(X)
        X=X[:,0]
        return self.mlp_head(X)
        

In [ ]:
epochs=20
learning_rate=3e-4

In [ ]:
transformation_operation=torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

In [ ]:
train_dataset=torchvision.datasets.MNIST(root="./data",train=True,download=True,transform=transformation_operation)
validation_dataset=torchvision.datasets.MNIST(root="./data",train=False,download=True,transform=transformation_operation)

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
validation_loader=DataLoader(validation_dataset,batch_size=32,shuffle=True)

In [ ]:
model=VisionTransformer(emb_dim,attention_heads,dropout_rate)

In [ ]:
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [ ]:


for epoch in range(epochs):
    epoch_accuracy=0
    total_correct=0
    total_points=0
    total_epoch_loss=0
    print("epoch:",epoch)
    for input,label in train_loader:

        output=model(input)
        loss_function=nn.CrossEntropyLoss()
        loss=loss_function(output,label.long())
         
        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        preds=output.argmax(axis=1)

        correct=(preds==label).sum().item()
        total_correct+=correct
        total_points+=input.size(0)
        
        accuracy=(correct/label.size(0))*100.0
        

        total_epoch_loss+=loss.item()
    epoch_accuracy=(total_correct/total_points)*100.0
    print("accuracy",epoch_accuracy)
    print(total_epoch_loss/len(train_loader))        


In [ ]:
#validation
with torch.no_grad():
    correct=0
    total=0
    for inputs,labels in validation_loader:
        
        output=model(inputs)
        result=output.argmax(axis=1)
        correct+=(result==labels).sum().item()
        total+=labels.size(0)
    
    accuracy=(correct/total)*100.0
    print("validation_accuracy:",accuracy)